# Global Living Conditions Agent Based Spatial Analysis


## Import Libraries and Configure Analysis Settings


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import shutil
import time
import urllib.error
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection, LineCollection
from matplotlib.patches import Polygon, Patch
from matplotlib.colors import Normalize
from scipy import stats
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from IPython.display import display

LOCAL_INPUT_FILES = []
START_YEAR = 2000
END_YEAR = 2023
N_NEIGHBOURS = 4
PERMUTATIONS = 9999
SIMULATIONS = 999
SENSITIVITY_RUNS = 199
ANNUAL_SUPPORT_RATE = 0.002
SEED = 20261107
SHOW_FIGURES = True
OUT = Path('global_living_conditions_spatial_outputs')
BASE = OUT
FIG = OUT / 'figures'
for folder in ['tables', 'figures', 'source', 'data']:
    (OUT / folder).mkdir(parents=True, exist_ok=True)

COLS = ['electricity_access_pct', 'clean_cooking_access_pct',
        'basic_water_access_pct', 'pm25_exposure_ug_m3']
WEIGHTS = ['population_total', 'population_urban']
META = ['country_code', 'country_name', 'iso2_code', 'year',
        'region_code', 'region', 'income_group_code', 'income_group',
        'classification_as_of']
LABELS = ['Electricity access', 'Clean cooking access',
          'Basic drinking water', 'Ambient PM2.5 exposure']

def save(name, frame):
    frame.to_csv(OUT / 'tables' / f'{name}.csv', index=False)

print(f'Years: {START_YEAR}–{END_YEAR}; neighbours: {N_NEIGHBOURS}')


## Upload and Clean the Annual Country Panel


In [ ]:
def read_panel_file(path):
    path = Path(path)
    options = dict(keep_default_na=False, na_values=[''])
    if path.suffix.lower() == '.csv':
        frame = pd.read_csv(path, **options)
    elif path.suffix.lower() in ['.xlsx', '.xls']:
        book = pd.ExcelFile(path)
        sheet = 'Panel' if 'Panel' in book.sheet_names else book.sheet_names[0]
        frame = pd.read_excel(book, sheet_name=sheet, **options)
    else:
        raise ValueError('Upload a CSV or Excel dataset.')
    frame.columns = frame.columns.astype(str).str.strip()
    return frame.replace(r'^\s*$', np.nan, regex=True)

def complete_countries(frame, variables, periods):
    keep_columns = [column for column in META if column in frame.columns] + variables + WEIGHTS
    required = ['country_code', 'country_name', 'year', 'region', 'income_group'] + variables + WEIGHTS
    absent = sorted(set(required) - set(frame.columns))
    if absent:
        raise ValueError(f'Missing required columns: {absent}')
    frame = frame.copy()
    frame['year'] = pd.to_numeric(frame['year'], errors='raise').astype(int)
    frame = frame.loc[frame.year.isin(periods), keep_columns].copy()
    if frame[['country_code', 'year']].isna().any().any():
        raise ValueError('Country codes and years cannot be missing.')
    frame['country_code'] = frame.country_code.astype(str).str.strip().str.upper()
    if frame.duplicated(['country_code', 'year']).any():
        raise ValueError('Duplicate country-year records must be resolved first.')
    for column in variables + WEIGHTS:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    audit_rows = []
    for country, group in frame.groupby('country_code', sort=True):
        numeric = group[variables + WEIGHTS]
        missing = group.isna().any().any()
        zero = numeric.eq(0).any().any()
        invalid = ((numeric < 0).any().any() or np.isinf(numeric.to_numpy(dtype=float)).any())
        complete_grid = len(group) == len(periods) and set(group.year) == set(periods)
        retained = not (missing or zero or invalid) and complete_grid
        audit_rows.append({'country_code': country, 'country_name': group.country_name.iloc[0],
                           'retained': retained, 'missing_data': bool(missing),
                           'zero_measurement': bool(zero), 'invalid_measurement': bool(invalid),
                           'complete_time_grid': bool(complete_grid)})
    audit = pd.DataFrame(audit_rows)
    eligible = audit.loc[audit.retained, 'country_code']
    clean = frame.loc[frame.country_code.isin(eligible)].sort_values(['country_code', 'year']).reset_index(drop=True)
    return clean, audit

if LOCAL_INPUT_FILES:
    input_paths = [Path(path) for path in LOCAL_INPUT_FILES]
else:
    from google.colab import files
    uploaded = files.upload()
    input_paths = [Path(name) for name in uploaded
                   if Path(name).suffix.lower() in ['.csv', '.xlsx', '.xls']]
    if not input_paths:
        raise ValueError('Upload your Global Living Conditions dataset.')

input_frames = [(path, read_panel_file(path)) for path in input_paths]
annual_candidates = []
for path, frame in input_frames:
    if {'year', 'country_code', *COLS, *WEIGHTS}.issubset(frame.columns):
        present_years = set(pd.to_numeric(frame.year, errors='coerce').dropna().astype(int))
        if set(range(START_YEAR, END_YEAR + 1)).issubset(present_years):
            annual_candidates.append((path, frame))
if not annual_candidates:
    raise ValueError('Upload the original panel or the annual 2000–2023 cleaned CSV.')
input_path, original = max(annual_candidates, key=lambda item: item[1].country_code.nunique())
if (START_YEAR, END_YEAR) != (2000, 2023):
    raise ValueError('The historical training and test windows require 2000–2023.')
panel, country_audit = complete_countries(original, COLS, list(range(START_YEAR, END_YEAR + 1)))
if panel.empty or panel.country_code.nunique() <= 8:
    raise ValueError('At least nine complete countries are required for these spatial checks.')
assert not panel.isna().any().any()
assert not panel.select_dtypes(include='number').eq(0).any().any()
for column in COLS[:3]:
    assert panel[column].between(0, 100).all()
panel.to_csv(OUT / 'data' / 'Annual_Clean_Panel.csv', index=False)
save('Country_Cleaning_Audit', country_audit)
print(f'{panel.country_code.nunique()} complete countries; {len(panel):,} annual records; four core indicators.')
display(panel.head())


## Load Country Coordinates and Map Boundaries


In [ ]:
COORDINATES_AS_OF = '2026-11-07'
COORDINATES_SOURCE = 'https://api.worldbank.org/v2/country?format=json&per_page=20000'
coords = {'ABW': (12.5167, -70.0167),
 'AFG': (34.5228, 69.1761),
 'AGO': (-8.81155, 13.242),
 'ALB': (41.3317, 19.8172),
 'AND': (42.5075, 1.5218),
 'ARE': (24.4764, 54.3705),
 'ARG': (-34.6118, -58.4173),
 'ARM': (40.1596, 44.509),
 'ASM': (-14.2846, -170.691),
 'ATG': (17.1175, -61.8456),
 'AUS': (-35.282, 149.129),
 'AUT': (48.2201, 16.3798),
 'AZE': (40.3834, 49.8932),
 'BDI': (-3.3784, 29.3639),
 'BEL': (50.8371, 4.36761),
 'BEN': (6.4779, 2.6323),
 'BFA': (12.3605, -1.53395),
 'BGD': (23.7055, 90.4113),
 'BGR': (42.7105, 23.3238),
 'BHR': (26.1921, 50.5354),
 'BHS': (25.0661, -77.339),
 'BIH': (43.8607, 18.4214),
 'BLR': (53.9678, 27.5766),
 'BLZ': (17.2534, -88.7713),
 'BMU': (32.3293, -64.706),
 'BOL': (-13.9908, -66.1936),
 'BRA': (-15.7801, -47.9292),
 'BRB': (13.0935, -59.6105),
 'BRN': (4.94199, 114.946),
 'BTN': (27.5768, 89.6177),
 'BWA': (-24.6544, 25.9201),
 'CAF': (5.63056, 21.6407),
 'CAN': (45.4215, -75.6919),
 'CHE': (46.948, 7.44821),
 'CHL': (-33.475, -70.6475),
 'CHN': (40.0495, 116.286),
 'CIV': (5.332, -4.0305),
 'CMR': (3.8721, 11.5174),
 'COD': (-4.325, 15.3222),
 'COG': (-4.2767, 15.2662),
 'COL': (4.60987, -74.082),
 'COM': (-11.6986, 43.2418),
 'CPV': (14.9218, -23.5087),
 'CRI': (9.63701, -84.0089),
 'CUB': (23.1333, -82.3667),
 'CYM': (19.3022, -81.3857),
 'CYP': (35.1676, 33.3736),
 'CZE': (50.0878, 14.4205),
 'DEU': (52.5235, 13.4115),
 'DJI': (11.5806, 43.1425),
 'DMA': (15.2976, -61.39),
 'DNK': (55.6763, 12.5681),
 'DOM': (18.479, -69.8908),
 'DZA': (36.7397, 3.05097),
 'ECU': (-0.229498, -78.5243),
 'EGY': (30.0982, 31.2461),
 'ERI': (15.3315, 38.9183),
 'ESP': (40.4167, -3.70327),
 'EST': (59.4392, 24.7586),
 'ETH': (9.02274, 38.7468),
 'FIN': (60.1608, 24.9525),
 'FJI': (-18.1149, 178.399),
 'FRA': (48.8566, 2.35097),
 'FRO': (61.8926, -6.91181),
 'FSM': (6.91771, 158.185),
 'GAB': (0.38832, 9.45162),
 'GBR': (51.5002, -0.126236),
 'GEO': (41.71, 44.793),
 'GHA': (5.57045, -0.20795),
 'GIN': (9.51667, -13.7),
 'GMB': (13.4495, -16.5885),
 'GNB': (11.8037, -15.1804),
 'GNQ': (3.7523, 8.7741),
 'GRC': (37.9792, 23.7166),
 'GRD': (12.0653, -61.7449),
 'GRL': (64.1836, -51.7214),
 'GTM': (14.6248, -90.5328),
 'GUM': (13.4443, 144.794),
 'GUY': (6.80461, -58.1548),
 'HKG': (22.3964, 114.109),
 'HND': (15.1333, -87.4667),
 'HRV': (45.8069, 15.9614),
 'HTI': (18.5392, -72.3288),
 'HUN': (47.4984, 19.0408),
 'IDN': (-6.19752, 106.83),
 'IMN': (54.1509, -4.47928),
 'IND': (28.6353, 77.225),
 'IRL': (53.3441, -6.26749),
 'IRN': (35.6878, 51.4447),
 'IRQ': (33.3302, 44.394),
 'ISL': (64.1353, -21.8952),
 'ISR': (31.7717, 35.2035),
 'ITA': (41.8955, 12.4823),
 'JAM': (17.9927, -76.792),
 'JOR': (31.9497, 35.9263),
 'JPN': (35.67, 139.77),
 'KAZ': (51.1879, 71.4382),
 'KEN': (-1.27975, 36.8126),
 'KGZ': (42.8851, 74.6057),
 'KHM': (11.5556, 104.874),
 'KIR': (1.32905, 172.979),
 'KNA': (17.3, -62.7309),
 'KOR': (37.5323, 126.957),
 'KWT': (29.3721, 47.9824),
 'LAO': (18.5826, 102.177),
 'LBN': (33.8872, 35.5134),
 'LBR': (6.30039, -10.7957),
 'LBY': (32.8578, 13.1072),
 'LCA': (14.0, -60.9832),
 'LIE': (47.1411, 9.52148),
 'LKA': (6.92148, 79.8528),
 'LSO': (-29.5208, 27.7167),
 'LTU': (54.6896, 25.2799),
 'LUX': (49.61, 6.1296),
 'LVA': (56.9465, 24.1048),
 'MAC': (22.1667, 113.55),
 'MAR': (33.9905, -6.8704),
 'MCO': (43.7325, 7.41891),
 'MDA': (47.0167, 28.8497),
 'MDG': (-20.4667, 45.7167),
 'MDV': (4.1742, 73.5109),
 'MEX': (19.427, -99.1276),
 'MHL': (7.11046, 171.135),
 'MKD': (42.0024, 21.4361),
 'MLI': (13.5667, -7.50034),
 'MLT': (35.9042, 14.5189),
 'MMR': (21.914, 95.9562),
 'MNE': (42.4602, 19.2595),
 'MNG': (47.9129, 106.937),
 'MNP': (15.1935, 145.765),
 'MOZ': (-25.9664, 32.5713),
 'MRT': (18.2367, -15.9824),
 'MUS': (-20.1605, 57.4977),
 'MWI': (-13.9899, 33.7703),
 'MYS': (3.12433, 101.684),
 'NAM': (-22.5648, 17.0931),
 'NCL': (-22.2677, 166.464),
 'NER': (13.514, 2.1073),
 'NGA': (9.05804, 7.48906),
 'NIC': (12.1475, -86.2734),
 'NLD': (52.3738, 4.89095),
 'NOR': (59.9138, 10.7387),
 'NPL': (27.6939, 85.3157),
 'NRU': (-0.5477, 166.920867),
 'NZL': (-41.2865, 174.776),
 'OMN': (23.6105, 58.5874),
 'PAK': (30.5167, 72.8),
 'PAN': (8.99427, -79.5188),
 'PER': (-12.0931, -77.0465),
 'PHL': (14.5515, 121.035),
 'PLW': (7.34194, 134.479),
 'PNG': (-9.47357, 147.194),
 'POL': (52.26, 21.02),
 'PRI': (18.23, -66.0),
 'PRK': (39.0319, 125.754),
 'PRT': (38.7072, -9.13552),
 'PRY': (-25.3005, -57.6362),
 'PYF': (-17.535, -149.57),
 'QAT': (25.2948, 51.5082),
 'ROU': (44.4479, 26.0979),
 'RUS': (55.7558, 37.6176),
 'RWA': (-1.95325, 30.0587),
 'SAU': (24.6748, 46.6977),
 'SDN': (15.5932, 32.5363),
 'SEN': (14.7247, -17.4734),
 'SGP': (1.28941, 103.85),
 'SLB': (-9.42676, 159.949),
 'SLE': (8.4821, -13.2134),
 'SLV': (13.7034, -89.2073),
 'SMR': (43.9322, 12.4486),
 'SOM': (2.07515, 45.3254),
 'SRB': (44.8024, 20.4656),
 'SSD': (4.85, 31.6),
 'STP': (0.20618, 6.6071),
 'SUR': (5.8232, -55.1679),
 'SVK': (48.1484, 17.1073),
 'SVN': (46.0546, 14.5044),
 'SWE': (59.3327, 18.0645),
 'SWZ': (-26.5225, 31.4659),
 'SYC': (-4.6309, 55.4466),
 'SYR': (33.5146, 36.3119),
 'TCA': (21.4602778, -71.141389),
 'TCD': (12.1048, 15.0445),
 'TGO': (6.1228, 1.2255),
 'THA': (13.7308, 100.521),
 'TJK': (38.5878, 68.7864),
 'TKM': (37.9509, 58.3794),
 'TLS': (-8.56667, 125.567),
 'TON': (-21.136, -175.216),
 'TTO': (10.6596, -61.4789),
 'TUN': (36.7899, 10.21),
 'TUR': (39.7153, 32.3606),
 'TUV': (-8.6314877, 179.089567),
 'TZA': (-6.17486, 35.7382),
 'UGA': (0.314269, 32.5729),
 'UKR': (50.4536, 30.5038),
 'URY': (-34.8941, -56.0675),
 'USA': (38.8895, -77.032),
 'UZB': (41.3052, 69.269),
 'VCT': (13.2035, -61.2653),
 'VEN': (9.08165, -69.8371),
 'VGB': (18.431389, -64.623056),
 'VIR': (18.3358, -64.8963),
 'VNM': (21.0069, 105.825),
 'VUT': (-17.7404, 168.321),
 'WSM': (-13.8314, -171.752),
 'XKX': (42.565, 20.926),
 'YEM': (15.352, 44.2075),
 'ZAF': (-25.746, 28.1871),
 'ZMB': (-15.3982, 28.2937),
 'ZWE': (-17.8312, 31.0672)}
REFERENCE_2023 = {'East Asia & Pacific': {'countries': 37, 'population_total': 2361068202},
 'Europe & Central Asia': {'countries': 58, 'population_total': 924805070},
 'Latin America & Caribbean': {'countries': 42, 'population_total': 657616519},
 'Middle East, North Africa, Afghanistan & Pakistan': {'countries': 23,
                                                       'population_total': 797801454},
 'North America': {'countries': 3, 'population_total': 376868838},
 'South Asia': {'countries': 6, 'population_total': 1662580579},
 'Sub-Saharan Africa': {'countries': 48, 'population_total': 1258809472}}
GEOGRAPHY_URL = 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/v5.1.2/geojson/ne_50m_admin_0_countries.geojson'
GEOGRAPHY_SHA256 = '3e458fc036ad0a66411f2c1e6cac49c5d7bfb81cb1123bc513b22511a2b7fdeb'
geometry_path = OUT / 'source' / 'natural_earth_50m_countries.geojson'
if not geometry_path.exists():
    for attempt in range(3):
        try:
            request = urllib.request.Request(GEOGRAPHY_URL, headers={'User-Agent': 'LivingConditionsResearch/1.0'})
            with urllib.request.urlopen(request, timeout=60) as response:
                geometry_bytes = response.read()
            geometry_path.write_bytes(geometry_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(attempt + 1)
assert hashlib.sha256(geometry_path.read_bytes()).hexdigest() == GEOGRAPHY_SHA256
geo = json.loads(geometry_path.read_text())
print(f'Capital coordinates: {COORDINATES_AS_OF}; map features: {len(geo["features"])}')


## Construct Geographic Neighbours and Spatial Weights


In [ ]:
def haversine(lat,lon):
    la=np.radians(lat);lo=np.radians(lon)
    dlat=la[:,None]-la[None,:];dlon=lo[:,None]-lo[None,:]
    d=2*6371.0088*np.arcsin(np.sqrt(np.clip(np.sin(dlat/2)**2+np.cos(la[:,None])*np.cos(la[None,:])*np.sin(dlon/2)**2,0,1)))
    np.fill_diagonal(d,np.inf)
    return d

def weights(d,k):
    nn=np.argsort(d,axis=1)[:,:k]
    w=np.zeros_like(d)
    w[np.arange(len(d))[:,None],nn]=1/k
    assert np.allclose(w.sum(axis=1),1) and np.allclose(np.diag(w),0)
    return w,nn


codes = sorted(panel.country_code.unique())
n = len(codes)
if not 1 <= N_NEIGHBOURS < n:
    raise ValueError('N_NEIGHBOURS must be smaller than the number of countries.')
years = np.arange(START_YEAR, END_YEAR + 1)
missing_coordinates = sorted(set(codes) - set(coords))
if missing_coordinates:
    raise ValueError(f'Coordinates unavailable for: {missing_coordinates}')
meta = panel.drop_duplicates('country_code').set_index('country_code').loc[codes]
meta = meta[['country_name', 'region', 'income_group']].copy()
meta['latitude'] = [coords[c][0] for c in codes]
meta['longitude'] = [coords[c][1] for c in codes]
agents = meta.copy()
save('Country_Agents', meta.reset_index())
d = haversine(meta.latitude.to_numpy(), meta.longitude.to_numpy())
w, nn = weights(d, N_NEIGHBOURS)
edge_rows = [{'country_code': codes[i], 'neighbour_code': codes[j],
              'weight': w[i, j], 'distance_km': d[i, j]}
             for i in range(n) for j in nn[i]]
save('Spatial_Edges', pd.DataFrame(edge_rows))
np.savez_compressed(OUT / 'source' / 'Agent_Network.npz', weights=w,
                    neighbours=nn, distances_km=d, country_codes=np.array(codes))
arrays = {column: panel.pivot(index='year', columns='country_code', values=column)
          .loc[years, codes].to_numpy() for column in COLS + ['population_total']}
pop = arrays['population_total'][-1]
regions = meta.region.to_numpy()
display(pd.DataFrame({'countries': [n], 'directed_links': [len(edge_rows)],
                      'weak_components': [connected_components(csr_matrix(w), directed=False)[0]],
                      'median_link_km': [np.median([row['distance_km'] for row in edge_rows])]}))


## Summarise Progress Income Groups and Population Coverage


In [ ]:
observed=[]
for year,sub in panel.groupby('year'):
    for col in COLS:
        v=sub[col].to_numpy();p=sub.population_total.to_numpy()
        observed.append({'year':year,'variable':col,'country_mean':v.mean(),'population_weighted_mean':np.average(v,weights=p),'country_SD':v.std(),'country_IQR':np.quantile(v,.75)-np.quantile(v,.25),'country_p10':np.quantile(v,.1),'country_p90':np.quantile(v,.9)})
save('Observed_Trends',pd.DataFrame(observed))
regional=[]
for (region,year),sub in panel.groupby(['region','year']):
    for col in COLS:
        regional.append({'region':region,'year':year,'variable':col,'countries':sub.country_code.nunique(),'country_mean':sub[col].mean(),'population_weighted_mean':np.average(sub[col],weights=sub.population_total)})
save('Regional_Trends',pd.DataFrame(regional))
income=[]
for (group,year),sub in panel.groupby(['income_group','year']):
    for col in COLS:income.append({'income_group':group,'year':year,'variable':col,'countries':sub.country_code.nunique(),'country_mean':sub[col].mean(),'population_weighted_mean':np.average(sub[col],weights=sub.population_total)})
save('Income_Trends',pd.DataFrame(income))

coverage = []
for region_name, reference in REFERENCE_2023.items():
    kept = panel.loc[panel.year.eq(2023) & panel.region.eq(region_name)]
    coverage.append({'region': region_name,
                     'original_countries': reference['countries'],
                     'retained_countries': len(kept),
                     'country_coverage_pct': 100 * len(kept) / reference['countries'],
                     'population_coverage_pct': 100 * kept.population_total.sum() / reference['population_total']})
save('Sample_Coverage', pd.DataFrame(coverage))
display(pd.read_csv(OUT / 'tables' / 'Observed_Trends.csv').query('year == 2023'))
display(pd.DataFrame(coverage))


## Measure Global Spatial Dependence Over Time


In [ ]:
def bh(p):
    p=np.asarray(p);order=np.argsort(p);q=np.empty(len(p))
    q[order]=np.minimum.accumulate((p[order]*len(p)/np.arange(1,len(p)+1))[::-1])[::-1]
    return np.minimum(q,1)

def moran(x,w,permutations=9999,seed=SEED):
    x=np.asarray(x,float);z=x-x.mean();den=z@z
    value=float(z@w@z/den)
    rng=np.random.default_rng(seed)
    order=np.argsort(rng.random((permutations,len(x))),axis=1)
    zp=z[order]
    sim=np.sum(zp*(zp@w.T),axis=1)/den
    expected=-1/(len(x)-1)
    p=(1+np.count_nonzero(np.abs(sim-expected)>=abs(value-expected)))/(permutations+1)
    return value,float(p)

morans=[]
for idx,col in enumerate(COLS):
    for j,year in enumerate(years):
        value,p=moran(arrays[col][j],w,PERMUTATIONS,SEED+idx*100+j)
        morans.append({'variable':col,'year':int(year),'Moran_I':value,'permutation_p':p})
morans=pd.DataFrame(morans);morans['BH_q']=bh(morans.permutation_p);save('Global_Moran_By_Year',morans)

display(morans.loc[morans.year.eq(2023)])


## Identify Adjusted Local Clusters and Outliers


In [ ]:
def local_moran(x,nn,permutations=9999,seed=SEED):
    n=len(x);z=(x-np.mean(x))/np.std(x);k=nn.shape[1]
    lag=z[nn].mean(axis=1);local=z*lag;rng=np.random.default_rng(seed);p=[]
    for i in range(n):
        choices=rng.integers(0,n-1,size=(permutations,k))
        duplicate=np.any(np.diff(np.sort(choices,axis=1),axis=1)==0,axis=1)
        while duplicate.any():
            choices[duplicate]=rng.integers(0,n-1,size=(duplicate.sum(),k))
            duplicate=np.any(np.diff(np.sort(choices,axis=1),axis=1)==0,axis=1)
        ids=choices+(choices>=i)
        perm=z[i]*z[ids].mean(axis=1)
        expected=-z[i]**2/(n-1)
        p.append((1+np.count_nonzero(abs(perm-expected)>=abs(local[i]-expected)))/(permutations+1))
    quadrant=np.where(z>=0,np.where(lag>=0,'High High','High Low'),np.where(lag>=0,'Low High','Low Low'))
    return pd.DataFrame({'z':z,'spatial_lag_z':lag,'local_I':local,'p':p,'quadrant':quadrant})

local=[]
for col,x in [('clean_cooking_deficit',100-arrays['clean_cooking_access_pct'][-1]),('pm25_exposure_ug_m3',arrays['pm25_exposure_ug_m3'][-1])]:
    frame=local_moran(x,nn,PERMUTATIONS,SEED);frame['country_code']=codes;frame['variable']=col;local.append(frame)
local=pd.concat(local,ignore_index=True);local['BH_q']=bh(local.p);local['cluster']=np.where(local.BH_q.le(.05),local.quadrant,'Not significant');save('Local_Moran_2023',local)

display(local.groupby(['variable', 'cluster']).size().rename('countries').reset_index())


## Test Alternative Geographic Neighbour Definitions


In [ ]:
sensitivity=[]
for k in [2,4,8]:
    wk,nk=weights(d,k)
    for col in COLS:
        value,p=moran(arrays[col][-1],wk,PERMUTATIONS,SEED)
        sensitivity.append({'neighbours':k,'variable':col,'Moran_I':value,'permutation_p':p})
save('Spatial_Weight_Sensitivity',pd.DataFrame(sensitivity))

display(pd.DataFrame(sensitivity))


## Compare Service Improvements and Air Quality Changes


In [ ]:
changes=meta.reset_index().copy()
for col in COLS:changes[col+'_change']=arrays[col][-1]-arrays[col][0]
changes['population_2023']=pop;save('Country_Changes_2000_2023',changes)
coupling=[]
for col in COLS[:3]:
    rho,p=stats.spearmanr(changes[col+'_change'],changes[COLS[3]+'_change'])
    coupling.append({'variable':col,'Spearman_change_vs_PM_change':rho,'descriptive_p':p})
save('Country_Change_Correlations',pd.DataFrame(coupling))

display(pd.DataFrame(coupling))


## Estimate Spatial Panel Associations and Diagnostics


In [ ]:
def twoway(x):return x-x.mean(axis=0,keepdims=True)-x.mean(axis=1,keepdims=True)+x.mean(axis=(0,1),keepdims=True)

def panel_slx(y,services,w,lags=3):
    # Predict current PM2.5 with previous-year own and neighbouring service access.
    t,n=y[1:].shape
    own=services[:-1]/10
    peer=np.einsum('ij,tjk->tik',w,own)
    x=np.concatenate([own,peer],axis=2)
    xd=twoway(x);yd=twoway(y[1:]);xf=xd.reshape(-1,6);yf=yd.reshape(-1)
    coef=np.linalg.lstsq(xf,yf,rcond=None)[0];res=(yf-xf@coef).reshape(t,n)
    bread=np.linalg.inv(xf.T@xf)
    scores=xd*res[:,:,None]
    group=scores.sum(axis=0)
    df_res=t*n-(n+t-1)-6
    cov_country=bread@(group.T@group)@bread*(n/(n-1))*((t*n-1)/df_res)
    gs=scores.sum(axis=1)
    meat=gs.T@gs
    for lag in range(1,lags+1):
        term=gs[lag:].T@gs[:-lag]
        meat+=(1-lag/(lags+1))*(term+term.T)
    # Same cluster-style small-sample factor as cov_nw_groupsum(use_correction='cluster').
    cov_dk=bread@meat@bread*(t/(t-1))*((t*n-1)/df_res)
    se=np.sqrt(np.maximum(np.diag(cov_dk),0));sec=np.sqrt(np.maximum(np.diag(cov_country),0))
    q=stats.t.ppf(.975,t-1)
    rows=[]
    names=['Own electricity','Own clean cooking','Own basic water','Neighbour electricity','Neighbour clean cooking','Neighbour basic water']
    for j,name in enumerate(names):
        rows.append({'term':name,'coefficient_per_10pp':coef[j],'DK_standard_error':se[j],'DK_low95':coef[j]-q*se[j],'DK_high95':coef[j]+q*se[j],'DK_p':2*stats.t.sf(abs(coef[j]/se[j]),t-1),'country_cluster_standard_error':sec[j],'country_cluster_p':2*stats.t.sf(abs(coef[j]/sec[j]),n-1)})
    vif=[]
    for j in range(6):
        other=np.delete(xf,j,axis=1);r=xf[:,j]-other@np.linalg.lstsq(other,xf[:,j],rcond=None)[0]
        vif.append(float((xf[:,j]@xf[:,j])/(r@r)))
    result=pd.DataFrame(rows);result['within_VIF']=vif
    serial=np.corrcoef(res[:-1].ravel(),res[1:].ravel())[0,1]
    hetero_x=np.column_stack([np.ones(len(xf)),xf]);sq=res.ravel()**2
    er=sq-hetero_x@np.linalg.lstsq(hetero_x,sq,rcond=None)[0]
    r2=1-er@er/np.sum((sq-sq.mean())**2)
    diagnostics={'observations':t*n,'countries':n,'years':t,'first_year':2001,'last_year':2023,'within_R2':float(1-np.sum(res**2)/np.sum(yd**2)),'max_within_VIF':max(vif),'within_condition_number':float(np.linalg.cond(xf/xf.std(axis=0))),'residual_lag1_correlation':float(serial),'BP_descriptive_LM':float(len(sq)*r2),'BP_descriptive_p':float(stats.chi2.sf(len(sq)*r2,6)),'BP_note':'Descriptive conventional LM only; dependence invalidates its textbook reference distribution. Robust covariance is primary.','DK_bandwidth':lags,'inference_df':t-1}
    return result,res,diagnostics

services=np.stack([arrays[c] for c in COLS[:3]],axis=2)
coefficients,residual,diagnostics=panel_slx(arrays[COLS[3]],services,w)
save('Spatial_SLX_Coefficients',coefficients)
residual_m=[]
for i,year in enumerate(years[1:]):
    v,p=moran(residual[i],w,PERMUTATIONS,SEED+i);residual_m.append({'year':int(year),'Moran_I':v,'permutation_p':p})
save('SLX_Residual_Spatial_Dependency',pd.DataFrame(residual_m))


display(coefficients)
display(pd.DataFrame([diagnostics]))


## Calibrate Country Agents and Validate Historical Predictions


In [ ]:
def fit_agent(a,nn,regions,beta,shrink):
    gap=np.maximum(a[:-1,nn].mean(axis=2)-a[:-1],0)
    innovations=np.diff(a,axis=0)-beta*gap
    raw=innovations.mean(axis=0);drift=raw.copy();length=len(innovations)
    for region in np.unique(regions):
        mask=regions==region;prior=innovations[:,mask].mean()
        drift[mask]=(length*raw[mask]+shrink*prior)/(length+shrink)
    residual=np.diff(a,axis=0)-drift[None,:]-beta*gap
    residual-=residual.mean(axis=0,keepdims=True)
    return drift,residual

def forecast(initial,drift,beta,nn,steps):
    values=[initial.copy()]
    for _ in range(steps):
        x=values[-1];peer=np.maximum(x[nn].mean(axis=1)-x,0)
        values.append(np.clip(x+drift+beta*peer,0,100))
    return np.asarray(values)

def errors(pred,actual,pop):
    err=pred-actual
    return {'RMSE_pp':float(np.sqrt(np.mean(err**2))),'MAE_pp':float(np.mean(abs(err))),'population_weighted_RMSE_pp':float(np.sqrt(np.average(err**2,weights=np.broadcast_to(pop,err.shape))))}

a=arrays['clean_cooking_access_pct'];calibration=[]
for beta in [0,.025,.05,.1,.2]:
    for shrink in [0,5,15]:
        drift,noise=fit_agent(a[:16],nn,regions,beta,shrink)
        prediction=forecast(a[15],drift,beta,nn,4)[1:]
        calibration.append({'beta':beta,'shrinkage_prior_years':shrink,**errors(prediction,a[16:20],pop)})
calibration=pd.DataFrame(calibration).sort_values(['RMSE_pp','beta','shrinkage_prior_years']);save('Agent_Validation_Grid_2016_2019',calibration)
chosen=calibration.iloc[0];beta=float(chosen.beta);shrink=float(chosen.shrinkage_prior_years)
drift,noise=fit_agent(a[:20],nn,regions,beta,shrink)
forecast_agent=forecast(a[19],drift,beta,nn,4)[1:]
nonspatial_shrink=float(calibration.loc[calibration.beta.eq(0)].iloc[0].shrinkage_prior_years)
nonspatial_drift,_=fit_agent(a[:20],nn,regions,0,nonspatial_shrink)
forecast_nonspatial=forecast(a[19],nonspatial_drift,0,nn,4)[1:]
drift_only=np.diff(a[:20],axis=0).mean(axis=0)
forecast_drift=forecast(a[19],drift_only,0,nn,4)[1:]
persistence=np.repeat(a[19][None,:],4,axis=0)
holdout=[]
for model,pred in [('Spatial country agents',forecast_agent),('Nonspatial country agents',forecast_nonspatial),('Country drift',forecast_drift),('Persistence',persistence)]:
    holdout.append({'model':model,**errors(pred,a[20:],pop)})
save('Agent_Heldout_2020_2023',pd.DataFrame(holdout))
validation=meta.reset_index().copy();validation['observed_2023']=a[-1];validation['agent_prediction_2023']=forecast_agent[-1];validation['nonspatial_prediction_2023']=forecast_nonspatial[-1];validation['drift_prediction_2023']=forecast_drift[-1];validation['absolute_agent_error_pp']=abs(forecast_agent[-1]-a[-1]);save('Agent_Heldout_Country_Errors',validation)


display(calibration.head())
display(pd.DataFrame(holdout))


## Simulate Conditional Clean Cooking Support Scenarios


In [ ]:
def allocate(x,pop,budget,score,cap_pp=5):
    cap=np.minimum(np.maximum(100-x,0),cap_pp)*pop[None,:]/100
    allocation=np.zeros_like(cap);remaining=np.full(len(x),budget)
    for _ in range(len(pop)+1):
        available=np.maximum(cap-allocation,0)
        effective=np.where(available>1e-6,np.maximum(score,0),0)
        total=effective.sum(axis=1)
        active=(remaining>1e-5)&(total>0)
        if not active.any():break
        proposal=np.zeros_like(cap)
        proposal[active]=remaining[active,None]*effective[active]/total[active,None]
        added=np.minimum(proposal,available);allocation+=added;remaining-=added.sum(axis=1)
    assert (allocation>=-1e-7).all() and (allocation<=cap+1e-4).all()
    assert (remaining>=-1e-3).all()
    return allocation*100/pop[None,:],remaining

def simulate(a,pop,nn,w,regions,beta,shrink,runs=999,budget_rate=.002,beta_scale=1,seed=SEED):
    drift,residual=fit_agent(a,nn,regions,beta,shrink)
    effective_beta=beta*beta_scale
    # Retain the empirical drift at the selected beta; beta_scale changes the interaction assumption.
    rng=np.random.default_rng(seed);n=len(pop);steps=7
    starts=rng.integers(0,len(residual),size=(runs,math.ceil(steps/3)))
    years=np.concatenate([(starts[:,j,None]+np.arange(3))%len(residual) for j in range(starts.shape[1])],axis=1)[:,:steps]
    shocks=residual[years]
    policies=['Baseline','Need proportional','Equity priority','Spatial targeting']
    low=a[-1]<=np.quantile(a[-1],.25)
    incoming=w.T@((100-a[-1])*pop/100)
    multiplier=1+incoming/(incoming.mean()+1e-12)
    outcomes={};unused={};budget=budget_rate*pop.sum()
    for policy in policies:
        states=np.repeat(a[-1][None,:],runs,axis=0);path=[states.copy()];unspent=np.zeros(runs)
        for step in range(steps):
            peer=np.maximum(states[:,nn].mean(axis=2)-states,0)
            next_states=np.clip(states+drift+effective_beta*peer+shocks[:,step,:],0,100)
            if policy!='Baseline':
                need=(100-next_states)*pop[None,:]/100
                if policy=='Need proportional':score=need
                elif policy=='Equity priority':score=need*low[None,:]
                else:score=need*multiplier[None,:]
                increments,left=allocate(next_states,pop,budget,score)
                next_states=np.clip(next_states+increments,0,100);unspent+=left
            states=next_states;path.append(states.copy())
        outcomes[policy]=np.stack(path,axis=1);unused[policy]=unspent
    baseline=outcomes['Baseline'][:,-1]
    summary=[];region_rows=[];paths=[]
    for policy,array in outcomes.items():
        final=array[:,-1];mean=final@pop/pop.sum();delta=(final-baseline)@pop/100
        equity=(final[:,low]@pop[low]/pop[low].sum())
        reached=(final>=90).sum(axis=1)
        summary.append({'policy':policy,'weighted_access_2030':mean.mean(),'access_low95':np.quantile(mean,.025),'access_high95':np.quantile(mean,.975),'additional_people_millions':delta.mean()/1e6,'additional_people_low95_m':np.quantile(delta,.025)/1e6,'additional_people_high95_m':np.quantile(delta,.975)/1e6,'baseline_low_quartile_access_2030':equity.mean(),'countries_above90_2030':reached.mean(),'unspent_support_millions':unused[policy].mean()/1e6})
        for k,year in enumerate(range(2023,2031)):
            p=array[:,k]@pop/pop.sum();paths.append({'policy':policy,'year':year,'mean':p.mean(),'low95':np.quantile(p,.025),'high95':np.quantile(p,.975)})
        for region in np.unique(regions):
            mask=regions==region
            region_rows.append({'policy':policy,'region':region,'weighted_access_2030':float((final[:,mask]@pop[mask]/pop[mask].sum()).mean()),'additional_people_millions':float(((final[:,mask]-baseline[:,mask])@pop[mask]/100).mean()/1e6)})
    return pd.DataFrame(summary),pd.DataFrame(paths),pd.DataFrame(region_rows),outcomes

scenarios,paths,policy_regions,outcomes=simulate(a,pop,nn,w,regions,beta,shrink,runs=SIMULATIONS,budget_rate=ANNUAL_SUPPORT_RATE)
save('Agent_Scenario_Summary_2030',scenarios);save('Agent_Scenario_Paths',paths);save('Agent_Scenario_Regions',policy_regions)
nonspatial_scenarios,_,_,_=simulate(a,pop,nn,w,regions,0,nonspatial_shrink,runs=SIMULATIONS,budget_rate=ANNUAL_SUPPORT_RATE)
save('Nonspatial_Agent_Scenario_Summary_2030',nonspatial_scenarios)
np.savez_compressed(OUT/'source/Agent_Scenario_Replicates.npz',**{k.replace(' ','_'):v for k,v in outcomes.items()},country_codes=np.array(codes))
country_final=meta.reset_index().copy()
for policy,values in outcomes.items():
    country_final[policy+'_mean_2030']=values[:,-1].mean(axis=0)
    country_final[policy+'_additional_pp']=(values[:,-1]-outcomes['Baseline'][:,-1]).mean(axis=0)
save('Agent_Country_Scenarios',country_final)

display(scenarios)
display(nonspatial_scenarios)


## Test Network Interaction and Policy Support Sensitivity


In [ ]:
sens=[]
for k in [2,4,8]:
    wk,nk=weights(d,k)
    for multiplier in [0,1,2]:
        result,_,_,_=simulate(a,pop,nk,wk,regions,beta,shrink,runs=SENSITIVITY_RUNS,beta_scale=multiplier)
        for _,row in result.iterrows():sens.append({'neighbours':k,'diffusion_multiplier':multiplier,**row.to_dict()})
save('Agent_Sensitivity',pd.DataFrame(sens))
budget_sens=[]
for rate in [.001,.002,.004]:
    result,_,_,_=simulate(a,pop,nn,w,regions,beta,shrink,runs=SENSITIVITY_RUNS,budget_rate=rate)
    for _,row in result.iterrows():budget_sens.append({'annual_support_fraction_of_population':rate,**row.to_dict()})
save('Agent_Budget_Sensitivity',pd.DataFrame(budget_sens))

display(pd.DataFrame(sens))
display(pd.DataFrame(budget_sens))


## Analyse Housing Deprivation When Data Are Available


In [ ]:
housing = None
housing_global = None
housing_candidates = [(path, frame) for path, frame in input_frames
                      if 'urban_slum_population_pct' in frame.columns]
if housing_candidates:
    housing_path, housing_source = max(housing_candidates, key=lambda item: item[1].country_code.nunique())
    housing, housing_audit = complete_countries(
        housing_source, COLS + ['urban_slum_population_pct'], list(range(2000, 2023, 2)))
    save('Housing_Country_Cleaning_Audit', housing_audit)
    if housing.empty or housing.country_code.nunique() <= 4:
        housing = None

if housing is not None:
    housing.to_csv(OUT / 'data' / 'Housing_Biennial_Clean_Panel.csv', index=False)
    hcodes = sorted(housing.country_code.unique())
    missing_coordinates = sorted(set(hcodes) - set(coords))
    if missing_coordinates:
        raise ValueError(f'Housing coordinates unavailable for: {missing_coordinates}')
    housing_last = housing.loc[housing.year.eq(2022)].set_index('country_code').loc[hcodes]
    hd = haversine(np.array([coords[c][0] for c in hcodes]),
                   np.array([coords[c][1] for c in hcodes]))
    hw, hnn = weights(hd, 4)
    hI, hp = moran(housing_last.urban_slum_population_pct.to_numpy(), hw, PERMUTATIONS, SEED)
    housing_global = {'countries': len(hcodes), 'Moran_I_2022': hI, 'permutation_p': hp}
    save('Housing_Global_Moran_2022', pd.DataFrame([housing_global]))
    housing_stats = []
    for column in COLS:
        rho, pvalue = stats.spearmanr(housing_last.urban_slum_population_pct, housing_last[column])
        housing_stats.append({'variable': column, 'Spearman_with_slum_share_2022': rho,
                              'descriptive_p': pvalue})
    save('Housing_Associations_2022', pd.DataFrame(housing_stats))
    display(pd.DataFrame([housing_global]))
    display(pd.DataFrame(housing_stats))
else:
    print('Housing analysis skipped: complete slum data were not uploaded.')


## Configure Scientific Charts and Geographic Map Rendering


In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':13,'axes.titlesize':15,'axes.labelsize':12,'xtick.labelsize':11,'ytick.labelsize':11,'legend.fontsize':10,'axes.spines.top':False,'axes.spines.right':False,'savefig.facecolor':'white'})
COLS=['electricity_access_pct','clean_cooking_access_pct','basic_water_access_pct','pm25_exposure_ug_m3']
LABELS=['Electricity access','Clean cooking access','Basic drinking water','Ambient PM2.5 exposure']
COLORS=['#267C69','#C56B2B','#2C6D9D','#7953A2']
POLICIES=['Baseline','Need proportional','Equity priority','Spatial targeting'];PCOL=['#5C6675','#2B73A3','#2B8B64','#B76331']
SHORT={'East Asia & Pacific':'East Asia and Pacific','Europe & Central Asia':'Europe and Central Asia','Latin America & Caribbean':'Latin America and Caribbean','Middle East, North Africa, Afghanistan & Pakistan':'MENA Afghanistan Pakistan','North America':'North America','South Asia':'South Asia','Sub-Saharan Africa':'Sub Saharan Africa'}

def table(name):return pd.read_csv(BASE/'tables'/f'{name}.csv')

def code_for(feature,values):
    p=feature['properties']
    for key in ['WB_A3','ISO_A3','ISO_A3_EH','ADM0_A3']:
        if p.get(key) in values:return p[key]
    return None

def rings(feature):
    g=feature['geometry'];coordinates=g['coordinates']
    polygons=[coordinates] if g['type']=='Polygon' else coordinates
    return [np.asarray(poly[0]) for poly in polygons]

def map_axis(ax,values,cmap='viridis',vmin=None,vmax=None,categories=None):
    patches=[];fills=[];matched=set()
    cmap_obj=plt.get_cmap(cmap)
    norm=Normalize(0,1) if categories is not None else Normalize(vmin if vmin is not None else min(values.values()),vmax if vmax is not None else max(values.values()))
    for feature in geo['features']:
        if feature['properties'].get('ADM0_A3')=='ATA':continue
        country=code_for(feature,values)
        color='#E5E8EB' if country is None else (categories[values[country]] if categories is not None else cmap_obj(norm(values[country])))
        if country is not None:matched.add(country)
        for ring in rings(feature):
            patches.append(Polygon(ring,closed=True));fills.append(color)
    collection=PatchCollection(patches,facecolors=fills,edgecolors='white',linewidths=.22,zorder=1)
    ax.add_collection(collection)
    for country in set(values)-matched:
        if country in agents.index:
            color=categories[values[country]] if categories is not None else cmap_obj(norm(values[country]))
            ax.scatter(agents.loc[country,'longitude'],agents.loc[country,'latitude'],s=15,c=[color],edgecolors='#445566',linewidths=.4,zorder=3)
    ax.set(xlim=(-180,180),ylim=(-58,84));ax.set_aspect('equal');ax.axis('off')
    if categories is None:return plt.cm.ScalarMappable(norm=norm,cmap=cmap_obj)

def finish(fig, name):
    fig.savefig(FIG / f'{name}.png', dpi=230, bbox_inches='tight')
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)

last = panel.loc[panel.year.eq(2023)].set_index('country_code')
trends = table('Observed_Trends')
region = table('Regional_Trends')
coverage = table('Sample_Coverage')
agents = meta.copy()
north_america = agents.loc[agents.region.eq('North America'), 'country_name'].tolist()
if len(north_america) == 1:
    SHORT['North America'] = f'North America {north_america[0]} only'


## Plot Geographic Sample and Population Coverage


In [ ]:
def plot_figure_01():
    fig,ax=plt.subplots(figsize=(9,4.6));y=np.arange(len(coverage));ax.barh(y,coverage.population_coverage_pct,color='#2C6D9D');ax.set_yticks(y,[SHORT[r] for r in coverage.region]);ax.invert_yaxis();ax.set(xlim=(0,112),xlabel='Retained share of regional population in 2023 (%)',title='Coverage differs sharply across regions')
    for j,row in coverage.iterrows():ax.text(row.population_coverage_pct+1.5,j,f'{row.population_coverage_pct:.1f}%',va='center',fontsize=11)
    ax.grid(axis='x',alpha=.2);fig.tight_layout();finish(fig,'01_sample_coverage')

plot_figure_01()


## Plot Observed Service and Pollution Trends


In [ ]:
def plot_figure_02():
    fig,axes=plt.subplots(2,2,figsize=(9,6.4),sharex=True)
    for j,(col,ax) in enumerate(zip(COLS,axes.ravel())):
        d=trends[trends.variable.eq(col)];ax.plot(d.year,d.population_weighted_mean,color=COLORS[j],lw=2.4,label='Population weighted');ax.plot(d.year,d.country_mean,color=COLORS[j],lw=1.7,ls='--',label='Country mean');ax.set_title(LABELS[j]);ax.set_ylabel('µg/m³' if j==3 else '% of population');ax.grid(alpha=.2)
        if j!=3:ax.set_ylim(35,101)
    axes[1,0].set_xlabel('Year');axes[1,1].set_xlabel('Year');axes[0,0].legend(frameon=False,fontsize=9,loc='lower right');fig.tight_layout();finish(fig,'02_observed_trends')

plot_figure_02()


## Map Clean Cooking and Ambient Pollution


In [ ]:
def plot_figure_03():
    fig,axes=plt.subplots(2,1,figsize=(9,7.4))
    for ax,col,title,cmap,vmax in [(axes[0],COLS[1],'Clean cooking access in 2023','YlGnBu',100),(axes[1],COLS[3],'Ambient PM2.5 exposure in 2023','YlOrRd',max(90,last[COLS[3]].max()))]:
        m=map_axis(ax,last[col].to_dict(),cmap,0,vmax);ax.set_title(title,loc='left');cb=fig.colorbar(m,ax=ax,orientation='horizontal',fraction=.045,pad=.015,shrink=.85);cb.set_label('% of population' if col==COLS[1] else 'µg/m³')
    fig.tight_layout();finish(fig,'03_maps_2023')

plot_figure_03()


## Compare Regional Clean Cooking Access Improvements


In [ ]:
def plot_figure_04():
    first=region[region.year.eq(2000)&region.variable.eq(COLS[1])].set_index('region');end=region[region.year.eq(2023)&region.variable.eq(COLS[1])].set_index('region')
    order=end.sort_values('population_weighted_mean').index;y=np.arange(len(order))
    fig,ax=plt.subplots(figsize=(9,4.5));ax.hlines(y,first.loc[order,'population_weighted_mean'],end.loc[order,'population_weighted_mean'],color='#BAC4CF',lw=3);ax.scatter(first.loc[order,'population_weighted_mean'],y,color='#A1AAB3',s=45,label='2000',zorder=3);ax.scatter(end.loc[order,'population_weighted_mean'],y,color=COLORS[1],s=58,label='2023',zorder=4);ax.set_yticks(y,[SHORT[r] for r in order]);ax.set(xlim=(0,108),xlabel='Population weighted clean cooking access (%)',title='Clean cooking progress remains uneven');ax.grid(axis='x',alpha=.2);ax.legend(frameon=False,loc='lower right');fig.tight_layout();finish(fig,'04_region_progress')

plot_figure_04()


## Plot Inequality Between Countries Over Time


In [ ]:
def plot_figure_05():
    fig,axes=plt.subplots(1,2,figsize=(9,3.7))
    for j,col in enumerate(COLS[:3]):
        d=trends[trends.variable.eq(col)];axes[0].plot(d.year,d.country_SD,color=COLORS[j],label=LABELS[j],lw=2)
    d=trends[trends.variable.eq(COLS[3])];axes[1].plot(d.year,d.country_SD,color=COLORS[3],lw=2)
    axes[0].set(title='Dispersion of service access',ylabel='Country standard deviation (pp)',xlabel='Year');axes[1].set(title='Dispersion of ambient PM2.5',ylabel='Country standard deviation (µg/m³)',xlabel='Year');axes[0].legend(frameon=False,fontsize=9);[ax.grid(alpha=.2) for ax in axes];fig.tight_layout();finish(fig,'05_country_dispersion')

plot_figure_05()


## Plot Global Spatial Dependence Over Time


In [ ]:
def plot_figure_06():
    moran=table('Global_Moran_By_Year');fig,ax=plt.subplots(figsize=(9,4.1))
    for j,col in enumerate(COLS):
        d=moran[moran.variable.eq(col)];ax.plot(d.year,d.Moran_I,lw=2.2,color=COLORS[j],label=LABELS[j])
    ax.set(xlabel='Year',ylabel="Moran's I",title='Geographic clustering persists across all four indicators');ax.grid(alpha=.2);ax.legend(frameon=False,ncol=2,loc='lower right');fig.tight_layout();finish(fig,'06_global_moran')

plot_figure_06()


## Map Adjusted Local Clusters and Outliers


In [ ]:
def plot_figure_07():
    local=table('Local_Moran_2023');categories={'High High':'#B84D43','Low Low':'#427CB1','High Low':'#DCAA53','Low High':'#67A6A2','Not significant':'#BFC8D1'}
    fig,axes=plt.subplots(2,1,figsize=(9,6.8))
    for ax,col,title in [(axes[0],'clean_cooking_deficit','Clusters of clean cooking deprivation'),(axes[1],COLS[3],'Clusters of high ambient PM2.5')]:
        v=local[local.variable.eq(col)].set_index('country_code').cluster.to_dict();map_axis(ax,v,categories=categories);ax.set_title(title,loc='left')
    handles=[Patch(color=categories[k],label=k) for k in ['High High','Low High','Not significant']]+[Patch(color='#E5E8EB',label='Outside sample')];fig.legend(handles=handles,loc='lower center',ncol=4,frameon=False,bbox_to_anchor=(.5,-.015),fontsize=9);fig.tight_layout();finish(fig,'07_local_clusters')

plot_figure_07()


## Plot Changes in Services and Pollution


In [ ]:
def plot_figure_08():
    changes=table('Country_Changes_2000_2023');fig,ax=plt.subplots(figsize=(9,4.6));palette=plt.cm.tab10(np.linspace(0,1,7))
    for j,reg in enumerate(sorted(changes.region.unique())):
        sub=changes[changes.region.eq(reg)];ax.scatter(sub[COLS[1]+'_change'],sub[COLS[3]+'_change'],s=20+np.sqrt(sub.population_2023)/550,c=[palette[j]],alpha=.75,label=SHORT[reg],edgecolors='white',linewidths=.5)
    for code in [c for c in ['PAK','IND','CHN','NGA','IDN','ETH'] if c in changes.country_code.values]:
        r=changes[changes.country_code.eq(code)].iloc[0];ax.annotate(code,(r[COLS[1]+'_change'],r[COLS[3]+'_change']),xytext=(5,4),textcoords='offset points',fontsize=10)
    ax.axhline(0,color='#73808D',lw=.8);ax.axvline(0,color='#73808D',lw=.8);ax.set(xlabel='Change in clean cooking access from 2000 to 2023 (pp)',ylabel='Change in ambient PM2.5 (µg/m³)',title='Service gains do not imply a uniform air quality response');ax.grid(alpha=.15);ax.legend(frameon=False,fontsize=9,loc='upper center',bbox_to_anchor=(.5,-.2),ncol=3);fig.tight_layout();finish(fig,'08_service_air_changes')

plot_figure_08()


## Map Country Agents and Geographic Connections


In [ ]:
def plot_figure_09():
    edges=table('Spatial_Edges');fig,ax=plt.subplots(figsize=(9,4.5));map_axis(ax,{},vmin=0,vmax=100)
    segments=[]
    for _,r in edges.iterrows():
        p=agents.loc[r.country_code];q=agents.loc[r.neighbour_code]
        if abs(p.longitude-q.longitude)<180:segments.append([(p.longitude,p.latitude),(q.longitude,q.latitude)])
    ax.add_collection(LineCollection(segments,colors='#8797AB',linewidths=.45,alpha=.45,zorder=2))
    sc=ax.scatter(agents.longitude,agents.latitude,c=last.loc[agents.index,COLS[1]],cmap='YlGnBu',vmin=0,vmax=100,s=18,edgecolors='white',linewidths=.3,zorder=3)
    ax.set_title(f'Country agents linked to {N_NEIGHBOURS} nearest retained capitals',loc='left');cb=fig.colorbar(sc,ax=ax,orientation='horizontal',fraction=.045,pad=.015,shrink=.85);cb.set_label('Observed clean cooking access in 2023 (%)');fig.tight_layout();finish(fig,'09_agent_network')

plot_figure_09()


## Compare Historical Agent Predictions and Benchmarks


In [ ]:
def plot_figure_10():
    errors=table('Agent_Heldout_Country_Errors');scores=table('Agent_Heldout_2020_2023');fig,axes=plt.subplots(1,2,figsize=(9,4))
    axes[0].scatter(errors.observed_2023,errors.agent_prediction_2023,s=20,alpha=.7,color='#2C6D9D');axes[0].plot([0,100],[0,100],color='#6D7683',ls='--');axes[0].set(xlabel='Observed access in 2023 (%)',ylabel='Predicted access in 2023 (%)',title='Four year held out predictions',xlim=(-2,102),ylim=(-2,102))
    model_labels=['Spatial agents','Nonspatial agents','Country drift','Persistence'];axes[1].barh(model_labels,scores.RMSE_pp,color=['#2B73A3','#77A6BC','#ABB9C6','#D1D8DE']);axes[1].invert_yaxis();axes[1].set(xlabel='RMSE over 2020 to 2023 (pp)',title='Comparison with simpler models');axes[1].set_xlim(0,3.2)
    for j,value in enumerate(scores.RMSE_pp):axes[1].text(value+.04,j,f'{value:.2f}',va='center',fontsize=11)
    fig.tight_layout();finish(fig,'10_agent_validation')

plot_figure_10()


## Plot Conditional Policy Simulation Paths


In [ ]:
def plot_figure_11():
    paths=table('Agent_Scenario_Paths');fig,ax=plt.subplots(figsize=(9,4.5))
    for policy,color in zip(POLICIES,PCOL):
        d=paths[paths.policy.eq(policy)];ax.plot(d.year,d['mean'],label=policy,color=color,lw=2.2);ax.fill_between(d.year,d.low95,d.high95,color=color,alpha=.07)
    ax.set(xlabel='Scenario year',ylabel='Population weighted clean cooking access (%)',title='Conditional scenarios through 2030');ax.grid(alpha=.2);ax.legend(frameon=False,fontsize=10,loc='upper left');fig.tight_layout();finish(fig,'11_agent_scenarios')

plot_figure_11()


## Compare Regional Gains Across Support Rules


In [ ]:
def plot_figure_12():
    inc=table('Agent_Scenario_Regions');order=sorted(inc.region.unique());fig,ax=plt.subplots(figsize=(9,4.4));y=np.arange(len(order))
    for j,(policy,color) in enumerate(zip(POLICIES[1:],PCOL[1:])):
        d=inc[inc.policy.eq(policy)].set_index('region').loc[order];ax.barh(y+(j-1)*.24,d.additional_people_millions,height=.22,label=policy,color=color)
    ax.set_yticks(y,[SHORT[r] for r in order]);ax.invert_yaxis();ax.set(xlabel='Additional covered 2023 population equivalents (millions)',title='The allocation rule changes who receives the gains');ax.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,-.2),ncol=3,fontsize=10);ax.grid(axis='x',alpha=.2);fig.tight_layout();finish(fig,'12_scenario_regions')

plot_figure_12()


## Plot Agent Interaction and Network Sensitivity


In [ ]:
def plot_figure_13():
    sens=table('Agent_Sensitivity');fig,axes=plt.subplots(1,2,figsize=(9,3.9),sharey=True)
    for ax,policy in zip(axes,['Equity priority','Spatial targeting']):
        matrix=sens[sens.policy.eq(policy)].pivot(index='neighbours',columns='diffusion_multiplier',values='additional_people_millions')
        im=ax.imshow(matrix,cmap='YlGnBu',aspect='auto',vmin=sens[sens.policy.ne('Baseline')].additional_people_millions.min(),vmax=sens.additional_people_millions.max())
        for i in range(3):
            for j in range(3):ax.text(j,i,f'{matrix.iloc[i,j]:.1f}',ha='center',va='center',fontsize=12,color='white' if im.norm(matrix.iloc[i,j])>.55 else 'black')
        ax.set_xticks(range(3),['0','1','2']);ax.set_yticks(range(3),['2','4','8']);ax.set(xlabel='Multiplier on calibrated diffusion',title=policy)
    axes[0].set_ylabel('Nearest neighbours');fig.subplots_adjust(wspace=.25,right=.84,bottom=.18,top=.85);cax=fig.add_axes([.88,.23,.025,.55]);fig.colorbar(im,cax=cax,label='Additional millions');finish(fig,'13_agent_sensitivity')

plot_figure_13()


## Map Housing Deprivation in the Complete Subset


In [ ]:
def plot_figure_14():
    if housing is None:
        return
    h=housing[housing.year.eq(2022)].set_index('country_code')
    fig,ax=plt.subplots(figsize=(9,4.1));m=map_axis(ax,h.urban_slum_population_pct.to_dict(),'YlOrRd',0,100);ax.set_title(f'Urban slum prevalence in the {housing.country_code.nunique()} country subset in 2022',loc='left');cb=fig.colorbar(m,ax=ax,orientation='horizontal',fraction=.045,pad=.015,shrink=.85);cb.set_label('% of urban population');fig.tight_layout();finish(fig,'14_housing_subset')

plot_figure_14()


## Export Analysis Tables Figures and Results


In [ ]:
summary = {
    'source_file': input_path.name,
    'input_sha256': hashlib.sha256(input_path.read_bytes()).hexdigest(),
    'countries': n,
    'annual_records': len(panel),
    'first_year': START_YEAR,
    'last_year': END_YEAR,
    'neighbours': N_NEIGHBOURS,
    'permutations': PERMUTATIONS,
    'primary_simulation_replicates': SIMULATIONS,
    'sensitivity_replicates': SENSITIVITY_RUNS,
    'agent_beta': beta,
    'agent_shrinkage_prior_years': shrink,
    'nonspatial_shrinkage_prior_years': nonspatial_shrink,
    'annual_support_fraction': ANNUAL_SUPPORT_RATE,
    'shock_block_length': 3,
    'country_classification_snapshot': COORDINATES_AS_OF,
    'reference_population_2023': sum(row['population_total'] for row in REFERENCE_2023.values()),
    'population_coverage_2023_pct': 100 * pop.sum() / sum(row['population_total'] for row in REFERENCE_2023.values()),
    'spatial_panel_diagnostics': diagnostics,
    'housing_analysis': housing_global,
    'interpretation': 'Conditional country-agent scenarios; no causal policy effects or household behaviour are identified.',
    'simulation_intervals': 'Process variation only; fitted parameters, population and unit support costs are held fixed.',
    'geometry_source': GEOGRAPHY_URL,
    'geometry_sha256': GEOGRAPHY_SHA256,
    'seed': SEED,
}
(OUT / 'Analysis_Settings.json').write_text(json.dumps(summary, indent=2))
archive_path = OUT.parent / 'Global_Living_Conditions_Spatial_Results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(OUT))
print(f'Saved {len(list(FIG.glob("*.png")))} figures and {len(list((OUT/"tables").glob("*.csv")))} result tables.')
try:
    from google.colab import files
except ImportError:
    print(archive_path.resolve())
else:
    files.download(str(archive_path))
